# ChatPromptTemplate的高级特性

### 1 部分变量预填充 partial()

举例：

In [2]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from rich import print as rprint

template = ChatPromptTemplate([
    {'role':'system','content':'你是{role},目标用户是{audience}'},
    {'role':'user','content':'{task}'}
])

prompt_result = template.invoke({'role':'导游','audience':'游客','task':'介绍一下北京的故宫'})
prompt_result2 = template.invoke({'role':'导游','audience':'游客','task':'介绍一下北京的颐和园'})

rprint(prompt_result)
rprint(prompt_result2)

ChatPromptValue(
    messages=[
        SystemMessage(content='你是导游,目标用户是游客', additional_kwargs={}, response_metadata={}),
        HumanMessage(content='介绍一下北京的故宫', additional_kwargs={}, response_metadata={})
    ]
)

ChatPromptValue(
    messages=[
        SystemMessage(content='你是导游,目标用户是游客', additional_kwargs={}, response_metadata={}),
        HumanMessage(content='介绍一下北京的颐和园', additional_kwargs={}, response_metadata={})
    ]
)

使用partial()优化

In [4]:
from langchain_core.prompts import ChatPromptTemplate
from rich import print as rprint

template = ChatPromptTemplate.from_messages([
    {'role': 'system', 'content': '你是{role},目标用户是{audience}'},
    {'role': 'user', 'content': '{task}'}
])

# 预填充变量(抽封公共常量)
final_template = template.partial(role="导游",audience="游客")

prompt_result = final_template.invoke({'task': '介绍一下北京的故宫'})
prompt_result2 = final_template.invoke({'task': '介绍一下北京的颐和园'})

rprint(prompt_result)
rprint(prompt_result2)

ChatPromptValue(
    messages=[
        SystemMessage(content='你是导游,目标用户是游客', additional_kwargs={}, response_metadata={}),
        HumanMessage(content='介绍一下北京的故宫', additional_kwargs={}, response_metadata={})
    ]
)

ChatPromptValue(
    messages=[
        SystemMessage(content='你是导游,目标用户是游客', additional_kwargs={}, response_metadata={}),
        HumanMessage(content='介绍一下北京的颐和园', additional_kwargs={}, response_metadata={})
    ]
)

## 2 消息占位符

### 2.1 使用placeholder

举例:

In [6]:
template = ChatPromptTemplate.from_messages([
    {'role':'system','content':'你是一个ai智能助手'},
    {'role':'placeholder','content':'{conversation}'}
])

result = template.invoke({
    'conversation':[
        {'role':'human','content':'你好请问明天天气如何？'},
        {'role':'ai','content':'明天天气晴朗'},
        {'role':'human','content':'后天天气如何？'}
    ]
})

rprint(result)

ChatPromptValue(
    messages=[
        SystemMessage(content='你是一个ai智能助手', additional_kwargs={}, response_metadata={}),
        HumanMessage(content='你好请问明天天气如何？', additional_kwargs={}, response_metadata={}),
        AIMessage(
            content='明天天气晴朗',
            additional_kwargs={},
            response_metadata={},
            tool_calls=[],
            invalid_tool_calls=[]
        ),
        HumanMessage(content='后天天气如何？', additional_kwargs={}, response_metadata={})
    ]
)

### 2.2 使用MessagesPlaceholder
举例:

In [7]:
from langchain_core.prompts import MessagesPlaceholder

template = ChatPromptTemplate.from_messages([
    {'role':'system','content':'你是一个ai智能助手'},
    MessagesPlaceholder(variable_name="conversation")
])

result = template.invoke({
    'conversation':[
        {'role':'human','content':'你好请问明天天气如何？'},
        {'role':'ai','content':'明天天气晴朗'},
        {'role':'human','content':'后天天气如何？'}
    ]
})

rprint(result)

ChatPromptValue(
    messages=[
        SystemMessage(content='你是一个ai智能助手', additional_kwargs={}, response_metadata={}),
        HumanMessage(content='你好请问明天天气如何？', additional_kwargs={}, response_metadata={}),
        AIMessage(
            content='明天天气晴朗',
            additional_kwargs={},
            response_metadata={},
            tool_calls=[],
            invalid_tool_calls=[]
        ),
        HumanMessage(content='后天天气如何？', additional_kwargs={}, response_metadata={})
    ]
)

## 3 模板库

In [10]:
class PromptLibrary:
    """可复用模板库"""

    TRANSLATOR = ChatPromptTemplate.from_messages([
        {'role':'system','content':'你是专业精通{source_lang}和{target_lang}'},
        {'role':'human','content':'翻译以下文本： \n{text}'}
    ])

    CODE_REVIEWER = ChatPromptTemplate.from_messages([
        {'role':'system','content':'你是{language}代码审查专家，重点关注{focus}'},
        {'role':'human','content':'审查代码： \n```{language}\n{code}\n```'}
    ])

    SUMMARIZER = ChatPromptTemplate.from_messages([
        {'role':'system','content':'你是内容摘要专家'},
        {'role':'human','content':'将以下内容总结为{num}个要点： \n{content}'}
    ])

    TUTOR = ChatPromptTemplate.from_messages([
        {'role':'system','content':'你是{subject}导师,学生水平{level}'},
        {'role':'human','content':'{question}'}
    ])

其他文件中调用

In [11]:
message = PromptLibrary.TRANSLATOR.format_messages(
    source_lang="英语",
    target_lang="中文",
    text="Hello World"
)

rprint(message)

[
    SystemMessage(content='你是专业精通英语和中文', additional_kwargs={}, response_metadata={}),
    HumanMessage(content='翻译以下文本： \nHello World', additional_kwargs={}, response_metadata={})
]